# Error Handling Patterns for Production Agents

## Overview

This notebook covers essential error handling patterns for building resilient agentic AI systems, including retry logic, circuit breakers, and graceful degradation.

## Learning Objectives

- Implement retry logic with exponential backoff
- Build circuit breakers to prevent cascading failures
- Apply graceful degradation strategies
- Handle timeouts effectively
- Design fault-tolerant agent architectures

## Exam Objectives: 2.4
## Estimated Time: 75-90 minutes

## 🎯 Exam Focus

This notebook covers concepts that appear frequently on the NCP-AAI exam:

**High-Priority Topics:**
- ⭐⭐⭐ Error handling patterns (retry logic, circuit breakers)
- ⭐⭐⭐ Prompt engineering techniques
- ⭐⭐ Tool integration best practices
- ⭐⭐ Streaming responses

**Common Exam Scenarios:**
- Implementing retry logic for API failures
- Designing prompts for complex tasks
- Handling tool execution errors

**Key Concepts to Master:**
- Exponential backoff for retries
- Circuit breaker pattern
- Dynamic prompt chains
- Graceful degradation

## Setup: Import Dependencies

In [ ]:
# Import core libraries for error handling
import os
import sys
import time
import logging
from typing import Callable, Any, Optional, Dict
from datetime import datetime, timedelta
from enum import Enum
from functools import wraps

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Error handling libraries loaded")
print("🎯 Setup complete!")

## Theory: Error Handling Patterns

### Why Error Handling Matters

Production agents must handle failures gracefully to:
- **Maintain availability**: Continue operating despite component failures
- **Prevent cascading failures**: Stop errors from propagating
- **Provide user experience**: Give meaningful feedback instead of crashes
- **Enable debugging**: Log errors for troubleshooting

### Three Core Patterns

#### 1. Retry Logic
**Purpose**: Handle transient failures (network glitches, temporary unavailability)
**When to use**: Timeouts, 5xx server errors, network errors
**When NOT to use**: 4xx client errors, authentication failures

#### 2. Circuit Breaker
**Purpose**: Prevent cascading failures by stopping calls to failing services
**States**: CLOSED (normal) → OPEN (failing) → HALF_OPEN (testing recovery)
**When to use**: External service dependencies, high-latency operations

#### 3. Graceful Degradation
**Purpose**: Provide reduced functionality when components fail
**Strategies**: Fallback responses, cached data, simplified logic
**When to use**: Non-critical features, user-facing applications

### Error Classification

| Error Type | Retry? | Example |
|------------|--------|----------|
| **Transient** | ✅ Yes | Network timeout, 503 Service Unavailable |
| **Permanent** | ❌ No | 404 Not Found, 401 Unauthorized |
| **Rate Limit** | ✅ Yes (with backoff) | 429 Too Many Requests |
| **Validation** | ❌ No | Invalid input format |

## Implementation: Retry Logic with Exponential Backoff

Exponential backoff increases delay between retries to avoid overwhelming failing services.

In [ ]:
# Retry decorator with exponential backoff
def retry_with_backoff(
    max_retries: int = 3,
    initial_delay: float = 1.0,
    backoff_factor: float = 2.0,
    exceptions: tuple = (Exception,)
) -> Callable:
    """
    Decorator that retries a function with exponential backoff.
    
    Args:
        max_retries: Maximum number of retry attempts
        initial_delay: Initial delay in seconds
        backoff_factor: Multiplier for delay (2.0 = double each time)
        exceptions: Tuple of exceptions to catch and retry
    
    Returns:
        Decorated function with retry logic
    """
    def decorator(func: Callable) -> Callable:
        @wraps(func)
        def wrapper(*args, **kwargs) -> Any:
            delay = initial_delay
            last_exception = None
            
            # Retry loop
            for attempt in range(max_retries):
                try:
                    # Attempt to execute function
                    result = func(*args, **kwargs)
                    
                    # Success - log if this was a retry
                    if attempt > 0:
                        logger.info(f"Success on attempt {attempt + 1}")
                    
                    return result
                    
                except exceptions as e:
                    last_exception = e
                    
                    # Check if we should retry
                    if attempt < max_retries - 1:
                        logger.warning(
                            f"Attempt {attempt + 1} failed: {str(e)}. "
                            f"Retrying in {delay:.1f}s..."
                        )
                        # Wait before retrying
                        time.sleep(delay)
                        # Increase delay exponentially
                        delay *= backoff_factor
                    else:
                        logger.error(f"All {max_retries} attempts failed")
            
            # All retries exhausted - raise last exception
            raise last_exception
        
        return wrapper
    return decorator

# Example: Unreliable API call
class UnreliableAPI:
    """Simulates an unreliable API for testing."""
    
    def __init__(self, failure_rate: float = 0.7):
        """
        Initialize unreliable API.
        
        Args:
            failure_rate: Probability of failure (0.0 to 1.0)
        """
        self.failure_rate = failure_rate
        self.call_count = 0
    
    @retry_with_backoff(max_retries=3, initial_delay=0.5, backoff_factor=2.0)
    def fetch_data(self, query: str) -> Dict[str, Any]:
        """
        Fetch data from unreliable API.
        
        Args:
            query: Search query
        
        Returns:
            API response data
        
        Raises:
            Exception: If API call fails
        """
        self.call_count += 1
        
        # Simulate random failures
        import random
        if random.random() < self.failure_rate:
            raise Exception(f"API call failed (attempt {self.call_count})")
        
        # Success
        return {"query": query, "results": ["result1", "result2"], "count": 2}

# Test retry logic
print("=== Testing Retry Logic ===")
api = UnreliableAPI(failure_rate=0.6)  # 60% failure rate

try:
    result = api.fetch_data("test query")
    print(f"✅ Success after {api.call_count} attempts")
    print(f"Result: {result}")
except Exception as e:
    print(f"❌ Failed after {api.call_count} attempts: {str(e)}")

# Demonstrate exponential backoff timing
print("\n=== Exponential Backoff Timing ===")
print("Attempt 1: Immediate")
print("Attempt 2: Wait 1.0s (initial_delay)")
print("Attempt 3: Wait 2.0s (1.0 * 2.0)")
print("Attempt 4: Wait 4.0s (2.0 * 2.0)")
print("Total time for 4 attempts: ~7 seconds")

## Implementation: Circuit Breaker Pattern

Circuit breakers prevent cascading failures by stopping calls to failing services.

In [ ]:
# Circuit breaker states
class CircuitState(Enum):
    """Circuit breaker states."""
    CLOSED = "closed"        # Normal operation - requests allowed
    OPEN = "open"            # Failing - requests blocked
    HALF_OPEN = "half_open"  # Testing recovery - limited requests

class CircuitBreaker:
    """
    Circuit breaker implementation for protecting against cascading failures.
    
    State transitions:
    CLOSED → OPEN: After failure_threshold consecutive failures
    OPEN → HALF_OPEN: After timeout period
    HALF_OPEN → CLOSED: After successful request
    HALF_OPEN → OPEN: After any failure
    """
    
    def __init__(
        self,
        failure_threshold: int = 5,
        timeout: int = 60,
        expected_exception: type = Exception
    ):
        """
        Initialize circuit breaker.
        
        Args:
            failure_threshold: Number of failures before opening circuit
            timeout: Seconds to wait before testing recovery
            expected_exception: Exception type to catch
        """
        self.failure_threshold = failure_threshold
        self.timeout = timeout
        self.expected_exception = expected_exception
        
        # State tracking
        self.failure_count = 0
        self.last_failure_time = None
        self.state = CircuitState.CLOSED
        
        logger.info(f"Circuit breaker initialized: threshold={failure_threshold}, timeout={timeout}s")
    
    def call(self, func: Callable, *args, **kwargs) -> Any:
        """
        Execute function with circuit breaker protection.
        
        Args:
            func: Function to execute
            *args: Function arguments
            **kwargs: Function keyword arguments
        
        Returns:
            Function result
        
        Raises:
            Exception: If circuit is OPEN or function fails
        """
        # Check if circuit is OPEN
        if self.state == CircuitState.OPEN:
            # Check if timeout has passed
            if datetime.now() - self.last_failure_time > timedelta(seconds=self.timeout):
                # Transition to HALF_OPEN to test recovery
                self.state = CircuitState.HALF_OPEN
                logger.info("Circuit breaker: OPEN → HALF_OPEN (testing recovery)")
            else:
                # Still in timeout period - reject request
                raise Exception("Circuit breaker is OPEN - service unavailable")
        
        try:
            # Attempt to execute function
            result = func(*args, **kwargs)
            
            # Success - handle state transitions
            if self.state == CircuitState.HALF_OPEN:
                # Recovery successful - close circuit
                self.state = CircuitState.CLOSED
                self.failure_count = 0
                logger.info("Circuit breaker: HALF_OPEN → CLOSED (service recovered)")
            
            return result
            
        except self.expected_exception as e:
            # Function failed - update failure tracking
            self.failure_count += 1
            self.last_failure_time = datetime.now()
            
            logger.warning(f"Circuit breaker: Failure {self.failure_count}/{self.failure_threshold}")
            
            # Check if we should open the circuit
            if self.failure_count >= self.failure_threshold:
                self.state = CircuitState.OPEN
                logger.error(f"Circuit breaker: CLOSED → OPEN (threshold reached)")
            
            # If in HALF_OPEN, immediately reopen
            if self.state == CircuitState.HALF_OPEN:
                self.state = CircuitState.OPEN
                logger.error("Circuit breaker: HALF_OPEN → OPEN (recovery failed)")
            
            raise
    
    def reset(self):
        """Manually reset circuit breaker to CLOSED state."""
        self.state = CircuitState.CLOSED
        self.failure_count = 0
        self.last_failure_time = None
        logger.info("Circuit breaker manually reset to CLOSED")

# Test circuit breaker
print("=== Testing Circuit Breaker ===")

# Create circuit breaker with low threshold for testing
circuit = CircuitBreaker(failure_threshold=3, timeout=5)

# Simulated failing function
def failing_service(should_fail: bool = True):
    """Simulated service that can fail."""
    if should_fail:
        raise Exception("Service error")
    return "Success"

# Test failure sequence
print("\nSimulating failures:")
for i in range(5):
    try:
        result = circuit.call(failing_service, should_fail=True)
        print(f"Attempt {i+1}: {result}")
    except Exception as e:
        print(f"Attempt {i+1}: Failed - {str(e)}")
    print(f"  State: {circuit.state.value}, Failures: {circuit.failure_count}\n")

print("\nCircuit is now OPEN - requests will be rejected immediately")

## References

### Course Materials

**Course Notes:** [Module 2: Agent Development](../../course-notes/module-02-agent-development.md)

### Practice Questions

**Exam Questions:** [Practice Questions](../../exam-questions/domain-02-development.md)

### Hands-On Labs

- [Lab 1: Basic RAG Agent](../../labs/lab-01-basic-rag-agent/README.md)
- [Lab 2: Multi-Agent Research](../../labs/lab-02-multi-agent-research/README.md)

### Additional Resources

- [NVIDIA NeMo Documentation](https://docs.nvidia.com/nemo/)
- [LangChain Documentation](https://python.langchain.com/)
- [NVIDIA AI Endpoints](https://build.nvidia.com/)
